# L'agent Data, pas à pas

Ce notebook refait **à la main, étape par étape**, ce que fait la fonction `fetch_prices` du fichier `data/data_agent.py`.
Il ne contient pas le code de l'agent : il l'**appelle**. Le vrai code reste dans `data/data_agent.py`.

Exécuter les cellules une par une avec **Shift + Entrée**.

## 0. Préparation

On indique à Python où se trouve le projet (le dossier au-dessus de `notebooks/`), pour pouvoir importer `data.data_agent`.
`autoreload` recharge automatiquement `data_agent.py` si on le modifie.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath(".."))

from datetime import date
import pandas as pd
from data.data_agent import load_yahoo_map, yahoo_download, fetch_prices

start, end = date(2025, 1, 1), date.today()

## 1. La table de correspondance

Belfius nous donne des noms (`WORLD`, `NASDAQ 3X`) et des ISIN, mais Yahoo Finance utilise ses propres symboles (`SXRT.DE`…).
Le fichier `data/etf_yahoo_map.csv` fait le lien. Il est créé par `python data/check_etfs_yfinance.py`,
qui cherche chaque ETF sur la bourse utilisée par Belfius (Xetra, sauf BEL 20 à Bruxelles et COPPER LVG / PALLADIUM LVG à Milan).

**À vérifier :** les 50 ETF sont là, avec un symbole et la devise `EUR`.

In [ ]:
ymap = load_yahoo_map()
print(len(ymap), "ETF dans la table")
ymap.head(10)

In [ ]:
# Devises des cotations retenues : idéalement, tout en EUR
ymap["currency"].value_counts()

## 2. Télécharger les prix bruts depuis Yahoo

Un seul appel pour les 50 ETF. On prend les **vrais cours de clôture**, sans ajustement des dividendes,
parce que Belfius valorise le portefeuille au vrai cours et ne crédite pas les dividendes.

**Résultat :** une ligne par jour, une colonne par symbole Yahoo.

In [ ]:
raw = yahoo_download(ymap["yahoo_symbol"].tolist(), start, end)
print(raw.shape[0], "jours ×", raw.shape[1], "symboles")
raw.head()

## 3. Aligner tous les ETF sur un même calendrier

Les bourses n'ont pas toutes les mêmes jours fériés. On met tous les ETF sur le calendrier des jours ouvrés,
et un jour sans cotation reprend le prix de la veille, **pendant 3 jours au maximum**.

**À vérifier :** il ne doit presque plus rester de cases vides (sauf au tout début, pour un ETF lancé récemment).

In [ ]:
calendrier = pd.bdate_range(start, end)
aligne = raw.reindex(calendrier).ffill(limit=3)

# Nombre de jours encore vides, pour les ETF les plus touchés
aligne.isna().sum().sort_values(ascending=False).head(10)

## 4. Renommer avec les noms Belfius

Dans tout le reste du projet, un ETF s'appelle par son nom Belfius (`WORLD`), jamais par son symbole Yahoo.

In [ ]:
noms = dict(zip(ymap["yahoo_symbol"], ymap["ticker"]))
prix = aligne.rename(columns=noms)
prix.tail()

## 5. Le résultat final de `fetch_prices`

La fonction complète fait les étapes 1 à 4 d'un coup, et met le tableau au format attendu par le reste de l'équipe :
**une ligne par jour et par ETF**, avec les colonnes `date`, `ticker`, `close_eur`.

**À vérifier :** le même nombre d'ETF qu'à l'étape 1, et les mêmes derniers prix qu'à l'étape 4.

In [ ]:
df = fetch_prices(start, end)
print(df["ticker"].nunique(), "ETF,", len(df), "lignes")
df.head()

In [ ]:
# Comparaison : les derniers prix de la fonction et ceux de l'étape 4 doivent être identiques
comparaison = pd.DataFrame({
    "fetch_prices": df.groupby("ticker")["close_eur"].last(),
    "etape_4": prix.ffill().iloc[-1].round(4),
})
comparaison["ecart"] = comparaison["fetch_prices"] - comparaison["etape_4"]
comparaison.sort_values("ecart", key=abs, ascending=False).head()